# **Trend Me** by Octobrain

**Trend Me** is a brand content generator tool based on context from trending videos.

---

***Execution***
Note: You require two API keys: *google-youtube api* and *hailouai*

1.   Initiate the colab with a GPU instance
2.   Run the cells in the current order
3.   Input the parameters indicated during the running.

---



***Pipeline***

1.   Search and download trending videos based on style, duration, and keywords.
2.   Extract contextual features from the videos including:
  * 'visual_style'
  *   'topic_summary'
  *   'text_narration'
  *  'visual_assets_needed'
  *  'audio_tone'
  *  'audio_type'
  *  'emotion_tone'
  *  'emotion_triggered'
  *  'trend_or_meme_reference'
  *  'target_audience'
  *  'audience_intent'

3. Build a video generation prompt based on contextual trending information and brand related user input
4. Generate a new video



# **Installing Needed Libraries**

In [ ]:
!pip install bitsandbytes
!pip install google-api-python-client yt_dlp isodate

In [ ]:
import os, cv2, tempfile
from moviepy.editor import *
import pandas as pd
import uuid
import requests
import cv2
import json
import torch
from transformers import LlavaNextVideoProcessor, LlavaNextVideoForConditionalGeneration

In [ ]:
from transformers import LlavaNextVideoProcessor, LlavaNextVideoForConditionalGeneration
from PIL import Image
import os
import json
import yt_dlp
import shutil
import isodate
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from google.colab import files
import isodate
from googleapiclient.discovery import build
from datetime import datetime, timedelta
import isodate

# **Youtube Scraper to extract trending videos**


In [ ]:


# Clave de API de YouTube (reemplázala con la tuya)
YOUTUBE_API_KEY = ""
COLAB_VIDEO_FOLDER = "/content/videos"

def customize_filters():
    """Allows users to modify default filtering keywords interactively."""
    default_positives = ["campaing", "trend", "aesthetic", "challenge", "reaction", "story time", "tutorial", "GRWM", "pov", "day in the life", "morning routine"]
    default_positives_dict = {1: "campaing", 2:"trend", 3:"aesthetic", 4:"challenge", 5:"reaction", 6:"story time", 7:"tutorial",
                              8:"GRWM", 9:"pov", 10:"day in the life", 11:"morning routine"}

    print("\n🛠️ Search Filter Configuration 🛠️")
    print("🔹 Positive Keywords (relevance):")
    print("   " + ", ".join(default_positives))

    category = int(input("\n¿Which categorie do you prefer? \n1.-campaing \n2.-trend \n3.-aesthetic \n4.-challenge \n5.-reaction \n6.-story time \n7.-tutorial \n8.-GRWM \n9.-pov \n10.-day in the life \n11.-morning routine:"))

    return default_positives_dict[category]# + default_negatives)


def get_start_of_week():
    today = datetime.utcnow()
    start_of_week = today - timedelta(days=today.weekday())  # Monday as start
    return start_of_week.replace(hour=0, minute=0, second=0, microsecond=0).isoformat("T") + "Z"

def search_youtube_shorts(query, max_results, min_duration, max_duration, custom_filters):
    """Searches for YouTube Shorts (videos ≤ 60s)."""
    youtube = build("youtube", "v3", developerKey=YOUTUBE_API_KEY)

    refined_query = f"{query} {custom_filters}".strip()
    print(f"\n🔍 Searching for YouTube Shorts with filter: {refined_query}\n")

    published_after = get_start_of_week()

    request = youtube.search().list(
        q=refined_query,
        part="snippet",
        regionCode="US",
        maxResults=max_results * 5,  # Grab more for filtering
        type="video",
        publishedAfter=published_after,
    )

    response = request.execute()
    video_ids = [item["id"]["videoId"] for item in response["items"]]

    if not video_ids:
        return []

    # Get video details including duration
    request = youtube.videos().list(
        part="contentDetails,snippet",
        id=",".join(video_ids)
    )
    response = request.execute()

    shorts = []
    for item in response["items"]:
        duration_iso = item["contentDetails"]["duration"]
        duration_seconds = int(isodate.parse_duration(duration_iso).total_seconds())

        if duration_seconds <= 30:  # Short format videos
            title = item["snippet"]["title"]
            description = item["snippet"]["description"]

            # Optional: further verify it's likely a "Short" by keywords
            if "shorts" in title.lower() or "shorts" in description.lower():
                video_data = {
                    "title": title,
                    "video_id": item["id"],
                    "url": f"https://www.youtube.com/watch?v={item['id']}",
                    "description": description,
                    "published_at": item["snippet"]["publishedAt"],
                    "duration_seconds": duration_seconds
                }
                shorts.append(video_data)

            if len(shorts) >= max_results:
                break

    return shorts



def download_video(video_url, output_folder="videos"):
    """Downloading a Youtube video as .mp4 and using yt-dlp."""
    os.makedirs(output_folder, exist_ok=True)
    ydl_opts = {
        "format": "best[ext=mp4]",
        "outtmpl": os.path.join(output_folder, "%(title)s.%(ext)s"),
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([video_url])

def zip_and_download():
    """Compresses all videos and metadata into a ZIP file and provides a download link."""
    zip_path = "/content/videos_download_brenda.zip"

    # Ensure metadata file exists before zipping
    metadata_path = os.path.join(COLAB_VIDEO_FOLDER, "videos_metadata.json")
    if not os.path.exists(metadata_path):
        print("⚠ Warning: Metadata file not found!")

    # Remove existing ZIP if it exists
    if os.path.exists(zip_path):
        os.remove(zip_path)

    # Create a ZIP file including videos and metadata
    shutil.make_archive(zip_path.replace(".zip", ""), 'zip', COLAB_VIDEO_FOLDER)

    # Download the ZIP file
    files.download(zip_path)


def extract_trend_video():
    """Main interactive function to fetch and download videos."""
    animal = "tiktok"
    num_videos = int(input("Introduce the number of videos you want to extract: "))
    min_duration = int(input("Minimum number of seconds (duration): "))
    max_duration = int(input("Maximum number of seconds (duration): "))

    # Allow user to customize filters dynamically
    custom_filters = customize_filters()

    print(f"\nSearching {num_videos} videos about '{animal}' in YouTube within {min_duration}-{max_duration} seconds...\n")

    videos = search_youtube_shorts(animal, max_results=num_videos, min_duration=min_duration, max_duration=max_duration, custom_filters=custom_filters)

    # Check if the number of videos found is less than requested
    if len(videos) < num_videos:
        print(f"\n⚠ There are only {len(videos)} videos found instead of {num_videos}.")
        adjust = input("¿Do you want to try with different filters? (s/n): ").strip().lower()
        if adjust == "s":
            return main()  # Restart the function with new filters

    if not videos:
        print("❌ No videos found with current parameters.")
        return

    # Mostrar resultados
    top_1 = None
    for idx, video in enumerate(videos):
        if idx == 0:
          top1= video['title']
        print(f"{idx + 1}. {video['title']} ({video['duration_seconds']} seconds)")
        print(f"   URL: {video['url']}")
        print(f"   Published on: {video['published_at']}\n")

    # Ensure the videos folder exists before saving metadata
    os.makedirs(COLAB_VIDEO_FOLDER, exist_ok=True)
    metadata_path = os.path.join(COLAB_VIDEO_FOLDER, "videos_metadata.json")
    with open(metadata_path, "w", encoding="utf-8") as f:
        json.dump(videos, f, indent=4, ensure_ascii=False)

    # Descargar los videos
    for video in videos:
        print(f"⬇️ Downloading: {video['title']}...")
        download_video(video["url"])
        print("✅ Download completed.\n")

    print("📁 Process finished. Video and metadata stored.")

    return os.path.join(COLAB_VIDEO_FOLDER, top1+".mp4")





In [ ]:
import shutil
import os


def clean_scraped_videos():
  #DELETES ALL VIDEOS AND METADATA FROM COLAB

  # Define the folder path
  video_folder = "/content/videos"

  # Remove all contents inside the folder
  shutil.rmtree(video_folder, ignore_errors=True)

  # Recreate the empty folder
  os.makedirs(video_folder, exist_ok=True)

  print("✅ Folder '/content/videos' has been cleared.")

  zipfile_path = "/content/videos_download.zip"
  if os.path.exists(zipfile_path):
      os.remove(zipfile_path)
      print("Zip file deleted successfully.")
  else:
      print("Zip file not found.")
  print(zipfile_path)

# **Video Language Model for Trending Context Extraction**

In [ ]:


def sample_video(video_path: str, video_out_path: str, sample_every = 30):
    print("sample every", sample_every)
    name = os.path.basename(video_path)
    base64Frames = []

    video = cv2.VideoCapture(video_path)
    width = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))

    idx_frame = 0
    samples = 0
    frame_list = []

    # Define the custom file name
    custom_filename = os.path.join(video_out_path, "{}_out.mp4".format(os.path.basename(video_path).split(".")[0]))

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Codec for mp4
    new_video_pose = cv2.VideoWriter(custom_filename, fourcc, 30.0, (width, height))

    while video.isOpened():
        success, frame = video.read()
        if not success:
            break

        if idx_frame % sample_every == 0:
            # Write the frame to the video
            pil_img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            frame_list.append(pil_img)

            new_video_pose.write(frame)
            samples +=1

        idx_frame += 1

    new_video_pose.release()
    print("Returning video with {} frames sampled".format(samples))
    return frame_list, custom_filename

In [ ]:

def load_video_analysis_model():
  device = "cuda" if torch.cuda.is_available() else "cpu"
  model_id = "llava-hf/LLaVA-NeXT-Video-7B-hf"

  model = LlavaNextVideoForConditionalGeneration.from_pretrained(
      model_id,
      torch_dtype=torch.float16,
      low_cpu_mem_usage=True,
      load_in_4bit=True
  ).to(device)

  processor = LlavaNextVideoProcessor.from_pretrained(model_id)
  return model, processor

In [ ]:

def process_video(model, processor, video_frames):
  conversation = [
      {
          "role": "user",
          "content": [
              {"type": "text",
              "text": """Analyze the following TikTok video and extract the key attributes that describe its format, style, and potential as viral or branded content.
              Be very descriptive and critical
              Return the results as a JSON list with keys and values:
                      {
                      "visual_style": "Describe the visual style: cinematic, selfie, vlog, animation, POV, sketch, etc.",
                      "topic_summary": "What is the main topic or message of the video?",
                      "text_narration": "Summarize the key text or narration lines, preferably segmented by scene or screen, if they are the same, just show one",
                      "visual_assets_needed": "What kind of images or clips could be used to represent this video? (e.g., cityscape, person typing, books, nature, etc.)",
                      "audio_tone": "What is the tone of the audio? (calm, upbeat, dramatic, humorous, etc.)",
                      "audio_type": "Type of audio: voice-over, trending music, original sound, sound effect, silence",
                      "emotion_tone": "Main emotional tone: humor, nostalgia, surprise, tenderness, motivation, etc.",
                      "emotion_triggered": "What emotion does this video evoke?",
                      "trend_or_meme_reference": "Does it reference any trend or meme? Describe which one specifically, try your better guess",
                      "target_audience": "Describe what kind of audience this video seems intended for",
                      "audience_intent": "What would the viewer gain from this video? e.g., knowledge, entertainment, emotion, motivation"
                    } """},
              {"type": "video"},
              ],
      },
  ]
  prompt = processor.apply_chat_template(conversation, add_generation_prompt=True)

  inputs = processor(text=prompt, videos=video_frames, padding=True, return_tensors="pt").to(model.device)

  output = model.generate(**inputs, max_new_tokens=10000, do_sample=False)

  output_text = processor.decode(output[0][2:], skip_special_tokens=True)

  clean_text = output_text.split("json")[1].replace("```", "")

  json_output = json.loads(clean_text)
  return json_output


In [ ]:
def analyse_videos(model, processor, video, max_frames = 5,  sample_rate = 50, out_path = COLAB_VIDEO_FOLDER):
  video_frames, _ =  sample_video(video, out_path, sample_rate)
  print("Processing video...")
  metadata = process_video(model, processor, video_frames)
  print("Key features extracted!")
  return metadata



# **Scraper and Context Extraction Steps**

In [ ]:
model, processor = load_video_analysis_model()

In [ ]:
clean_scraped_videos()
video = extract_trend_video()
metadata = analyse_videos(model, processor, video, sample_rate = 30)
metadata

In [ ]:
brand_or_product = input("Talk about your brand or product: ")

# **Video Generation based on trending context extraction**
# **API CALL TO : https://hailuoai.video/create**

In [ ]:
import os
import time
import requests
import json


api_key = ""

prompt = f"""Objective: Generate a compelling video  designed for viral potential, specifically targeting the current trend associated with the hashtag: {metadata["topic_summary"]}.

Core Strategy: Replicate key visual and conceptual elements identified in the top-performing video currently trending under # {metadata["topic_summary"]}.

Input Concepts (Derived from captioning/analysis of the top trending video):

visual style: {metadata["visual_style"]}
text narration: {metadata["text_narration"]}
visual assets_needed: {metadata["visual_assets_needed"]}
audio tone: {metadata["audio_tone"]}
audio type: {metadata["audio_type"]}
emotion tone: {metadata["emotion_tone"]}
emotion triggered: {metadata["emotion_triggered"]}
trend or meme reference: {metadata["trend_or_meme_reference"]}
target audience: {metadata["target_audience"]}
audience intent: {metadata["audience_intent"]}

Instructions for AI:

Incorporate Elements: Weave the listed Input Concepts naturally into the video's narrative, scenes, or visual focus.

Capture Trend Essence: Analyze the implied style, pacing, mood, camera angles (if applicable), and overall aesthetic of content typically found under #{metadata["topic_summary"]}.
Emulate these qualities in the generated video.

Viral Appeal: Optimize for engagement. The video should be visually interesting, potentially surprising or satisfying, and encourage sharing.
Focus on creating a strong hook within the first few seconds.

Brand Integration: Subtly and naturally incorporate the specified brand/product: {brand_or_product}.
Ensure the integration aligns with the trend's style and the video's core concepts, avoiding overly aggressive or out-of-place advertising.
The product/brand should feel like part of the scene or solution presented

Output: A video file incorporating the specified elements and targeting the identified trend."""

model = "T2V-01"
output_file_name = f"generated_video.mp4" #Please enter the save path for the generated video here

def invoke_video_generation()->str:
    print("-----------------Submit video generation task-----------------")
    url = "https://api.minimaxi.chat/v1/video_generation"
    payload = json.dumps({
      "prompt": prompt,
      "model": model
    })
    headers = {
      'authorization': 'Bearer ' + api_key,
      'content-type': 'application/json',
    }

    response = requests.request("POST", url, headers=headers, data=payload)
    print(response.text)
    task_id = response.json()['task_id']
    print("Video generation task submitted successfully, task ID.: "+task_id)
    return task_id

def query_video_generation(task_id: str):
    url = "https://api.minimaxi.chat/v1/query/video_generation?task_id="+task_id
    headers = {
      'authorization': 'Bearer ' + api_key
    }
    response = requests.request("GET", url, headers=headers)
    status = response.json()['status']
    if status == 'Preparing':
        print("...Preparing...")
        return "", 'Preparing'
    elif status == 'Queueing':
        print("...In the queue...")
        return "", 'Queueing'
    elif status == 'Processing':
        print("...Generating...")
        return "", 'Processing'
    elif status == 'Success':
        return response.json()['file_id'], "Finished"
    elif status == 'Fail':
        return "", "Fail"
    else:
        return "", "Unknown"


def fetch_video_result(file_id: str):
    print("---------------Video generated successfully, downloading now---------------")
    url = "https://api.minimaxi.chat/v1/files/retrieve?file_id="+file_id
    headers = {
        'authorization': 'Bearer '+api_key,
    }

    response = requests.request("GET", url, headers=headers)
    print(response.text)

    download_url = response.json()['file']['download_url']
    print("Video download link: " + download_url)
    with open(output_file_name, 'wb') as f:
        f.write(requests.get(download_url).content)
    print("THe video has been downloaded in："+os.getcwd()+'/'+output_file_name)


if __name__ == '__main__':
    task_id = invoke_video_generation()
    print("-----------------Video generation task submitted -----------------")
    while True:
        time.sleep(10)

        file_id, status = query_video_generation(task_id)
        if file_id != "":
            fetch_video_result(file_id)
            print("---------------Successful---------------")
            break
        elif status == "Fail" or status == "Unknown":
            print("---------------Failed---------------")
            break